# Transpilación de 1D-CNN-LSTM a TensorFlow Lite Micro (Cuantización INT8)

Este cuaderno carga el modelo entrenado Keras de Deep Learning de MyoTensor, y realiza la cuantización de enteros de 8 bits completa (Full Integer Quantization) utilizando un representative dataset para generar los archivos `NN_model.h` y `NN_model.cpp` directamente en el firmware del Clasificador.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Forzar ejecución en CPU para evitar ops GPU-only como CudnnRNNV3
import tensorflow as tf
import numpy as np
from pathlib import Path
from tensorflow.keras.layers import Layer
from tensorflow.keras import layers, models

# Cargar variables de entorno desde .env
def load_env_variables():
    start_dir = Path(os.getcwd())
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env")
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
    print(f"✅ .env cargado con éxito desde: {env_path}")

load_env_variables()

2026-06-07 16:05:46.957196: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-07 16:05:47.032438: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-06-07 16:05:48.824882: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


✅ .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


In [2]:
# Definición del AttentionLayer para poder deserializar el modelo entrenado
# Se utiliza operaciones estándar de TF en lugar de keras backend para mayor compatibilidad con TFLite
@tf.keras.utils.register_keras_serializable()
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)
    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1), initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(1,), initializer="zeros", trainable=True)
        super(AttentionLayer, self).build(input_shape)
    def call(self, x):
        e = tf.math.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = x * a
        return tf.reduce_sum(output, axis=1)
    def get_config(self): 
        return super(AttentionLayer, self).get_config()

In [3]:
# Cargar datos de entrenamiento para representative dataset
train_data_path = os.path.join(os.environ["PROCESSED_TENSOR_PROTO"], "X_train.npy")
print(f"Cargando X_train desde: {train_data_path}")
X_train = np.load(train_data_path)
print(f"Dimensiones de X_train: {X_train.shape}")

Cargando X_train desde: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/datasets/processed/myotensor_proto/tensor/X_train.npy
Dimensiones de X_train: (3272, 300, 1)


In [4]:
# Definir representative dataset generator
def representative_dataset():
    # Extraer 250 muestras aleatorias
    np.random.seed(42)
    indices = np.random.choice(X_train.shape[0], size=250, replace=False)
    for idx in indices:
        sample = X_train[idx].astype(np.float32)
        if sample.ndim == 1:
            sample = np.expand_dims(sample, axis=-1)
        elif sample.ndim == 2 and sample.shape[-1] != 1:
            sample = np.expand_dims(sample, axis=-1)
        # Añadir la dimensión de batch (1, 300, 1)
        yield [np.expand_dims(sample, axis=0)]

print("✅ Generador representative_dataset configurado.")

✅ Generador representative_dataset configurado.


In [5]:
# Cargar modelo Keras entrenado
model_path = os.path.join(os.environ["MODELS_DL_PROTO"], "myotensor_proto_net_trained.keras")
print(f"Cargando modelo Keras desde: {model_path}")
loaded_model = tf.keras.models.load_model(model_path, custom_objects={'AttentionLayer': AttentionLayer})
loaded_model.summary()

# Reconstruir el modelo con batch_shape estática y unrolling de LSTM para evitar fallos de cuantización
# (El unrolling de LSTM elimina loops dinámicos en el converter TFLite y optimiza la ejecución en el ESP32-S3)
print("Reconstruyendo modelo con entrada estática (batch_shape=(1, 300, 1)) y LSTM unrolled...")

def residual_inception_block(input_tensor, filters):
    branch1 = layers.Conv1D(filters, 3, padding='same', activation='relu')(input_tensor)
    branch2 = layers.Conv1D(filters, 5, padding='same', activation='relu')(input_tensor)
    branch3 = layers.Conv1D(filters, 9, padding='same', activation='relu')(input_tensor)
    concat = layers.concatenate([branch1, branch2, branch3], axis=-1)
    projection = layers.Conv1D(filters * 3, 1, padding='same')(input_tensor)
    x = layers.add([concat, projection])
    x = layers.Activation('relu')(x)
    x = layers.BatchNormalization()(x)
    return x

def build_static_model(batch_size=1, input_shape=(300, 1), num_classes=4):
    inputs = layers.Input(batch_shape=(batch_size, input_shape[0], input_shape[1]))
    x = layers.Conv1D(16, 3, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.2)(x)
    x = residual_inception_block(x, 16)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Dropout(0.3)(x)
    # LSTM unrolled
    x = layers.LSTM(32, return_sequences=True, unroll=True)(x)
    x = AttentionLayer()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(16, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs=inputs, outputs=outputs, name="MS_CLSTM_Static_Unrolled")
    return model

static_model = build_static_model(batch_size=1, input_shape=(300, 1), num_classes=4)

# Copiar pesos por coincidencia de formas para mapear correctamente las capas
print("Copiando pesos por coincidencia de formas...")
loaded_layers_with_weights = [l for l in loaded_model.layers if l.get_weights()]
static_layers_with_weights = [l for l in static_model.layers if l.get_weights()]
matched_loaded = set()
for s_layer in static_layers_with_weights:
    s_shapes = [w.shape for w in s_layer.get_weights()]
    for l_layer in loaded_layers_with_weights:
        if l_layer in matched_loaded:
            continue
        l_shapes = [w.shape for w in l_layer.get_weights()]
        if l_shapes == s_shapes:
            s_layer.set_weights(l_layer.get_weights())
            matched_loaded.add(l_layer)
            print(f"  Pesos copiados: {l_layer.name} ({l_shapes}) -> {s_layer.name}")
            break
print("✅ Copia de pesos finalizada con éxito.")

# Configurar conversor TFLite para cuantización entera INT8 completa
print("Convirtiendo modelo Keras a TFLite con cuantización entera INT8 completa...")
converter = tf.lite.TFLiteConverter.from_keras_model(static_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
print(f"✅ Conversión finalizada. Tamaño del modelo cuantizado: {len(tflite_model)} bytes")

# Guardar temporalmente en disco
tflite_save_path = os.path.join(os.environ["MODELS_DL_PROTO"], "myotensor_proto_net.tflite")
with open(tflite_save_path, "wb") as f:
    f.write(tflite_model)
print(f"✅ Modelo guardado en: {tflite_save_path}")

Cargando modelo Keras desde: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/models/myotensor_proto/dl/myotensor_proto_net_trained.keras


2026-06-07 16:05:51.847387: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-06-07 16:05:51.847431: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES=""
2026-06-07 16:05:51.847439: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to an empty string - this hides all GPUs from CUDA
2026-06-07 16:05:51.847444: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2026-06-07 16:05:51.847449: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: cbeLOQdebian
2026-06-07 16:05:51.847453: I external/local_xla/xla/stream_exe

Model: "MS_CLSTM_TinyML"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 300, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 300, 16)   │         64 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 150, 16)   │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 150, 16)   │          0 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 150, 16)   │        784 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 150, 16)   │      1,296 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 150, 16)   │      2,320 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 150, 48)   │          0 │ conv1d_1[0][0],   │
│ (Concatenate)       │                   │            │ conv1d_2[0][0],   │
│                     │                   │            │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 150, 48)   │        816 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 150, 48)   │          0 │ concatenate[0][0… │
│                     │                   │            │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 150, 48)   │          0 │ add[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 150, 48)   │        192 │ activation[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 75, 48)    │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 75, 48)    │          0 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 75, 32)    │     10,368 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer     │ (None, 32)        │         33 │ lstm[0][0]        │
│ (AttentionLayer)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32)        │          0 │ attention_layer[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │        528 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16)        │         64 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 4)         │         68 │ batch_normalizat

 Total params: 49,345 (192.76 KB)

...

INFO:tensorflow:Assets written to: /tmp/tmpa6592k55/assets


Saved artifact at '/tmp/tmpa6592k55'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 300, 1), dtype=tf.float32, name='keras_tensor_41')
Output Type:
  TensorSpec(shape=(1, 4), dtype=tf.float32, name=None)
Captures:
  140435264500160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140435264498400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140435265103376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140435265097568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140435265106192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140435265103904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140435265109712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140435265107072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140435265104432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140435233508864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140435233506224: Tenso

/home/cbe/miniconda3/envs/tesis_env/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1780862754.803801   48996 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1780862754.803827   48996 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-06-07 16:05:54.804232: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpa6592k55
2026-06-07 16:05:54.807521: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-06-07 16:05:54.807534: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpa6592k55
I0000 00:00:1780862754.850819   48996 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-06-07 16:05:54.855780: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-06

fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
2026-06-07 16:05:57.054074: I tensorflow/compiler/mlir/lite/flatbuffer_export.cc:4150] Estimated count of arithmetic ops: 3.958 M  ops, equivalently 1.979 M  MACs
2026-06-07 16:05:57.054094: W tensorflow/compiler/mlir/lite/flatbuffer_export.cc:3705] Skipping runtime version metadata in the model. This will be generated by the exporter.


✅ Conversión finalizada. Tamaño del modelo cuantizado: 589184 bytes
✅ Modelo guardado en: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/models/myotensor_proto/dl/myotensor_proto_net.tflite


In [6]:
# Generar archivos C++ (Header y Source) para PlatformIO
classifier_root = Path(os.environ["PROJECT_ROOT"]) / "firmware" / "Classifier"
header_path = classifier_root / "include" / "NN_model.h"
source_path = classifier_root / "src" / "NN_model.cpp"

# 1. Escribir Header (.h)
header_content = """#ifndef NN_MODEL_H
#define NN_MODEL_H

// Declaración externa del modelo cuantizado TFLite Micro
extern const unsigned char g_model_data[];
extern const int g_model_data_len;

#endif // NN_MODEL_H
"""

with open(header_path, "w", encoding="utf-8") as f:
    f.write(header_content)
print(f"✅ Archivo de cabecera creado: {header_path}")

# 2. Formatear bytes a hexadecimal para C++
c_hex_array = []
for val in tflite_model:
    c_hex_array.append(f"0x{val:02X}")

# Unir con comas y formatear con saltos de línea cada 12 elementos
formatted_array = ""
for i in range(0, len(c_hex_array), 12):
    formatted_array += "  " + ", ".join(c_hex_array[i:i+12]) + ",\n"
formatted_array = formatted_array.rstrip(",\n") + "\n"

# 3. Escribir Source (.cpp) con alineación a 16 bytes para habilitar instrucciones vectoriales en ESP32-S3
source_content = f"""#include \"NN_model.h\"

// Alineación requerida por TensorFlow Lite Micro en ESP32-S3
alignas(16) const unsigned char g_model_data[] = {{
{formatted_array}}};

const int g_model_data_len = {len(tflite_model)};
"""

with open(source_path, "w", encoding="utf-8") as f:
    f.write(source_content)
print(f"✅ Archivo de definición creado: {source_path}")

# 4. Generar archivo cnn_scaler_params.h dinámicamente desde el StandardScaler de entrenamiento
import joblib
scaler_path = Path(os.environ["PROCESSED_TENSOR_PROTO"]) / "std_scaler.bin"
if not scaler_path.exists():
    raise FileNotFoundError(
        f"⚠️ No se encontró el archivo del StandardScaler en: {scaler_path}.\n"
        f"Por favor, ejecuta primero el cuaderno de preprocesamiento `procesar_myotensor_proto.ipynb`."
    )

scaler = joblib.load(scaler_path)
signal_mean = scaler.mean_[0]
signal_std = scaler.scale_[0]

cnn_scaler_path = classifier_root / "include" / "cnn_scaler_params.h"
cnn_scaler_content = f"""#ifndef CNN_SCALER_PARAMS_H\n#define CNN_SCALER_PARAMS_H\n\n// Parámetros del StandardScaler para la señal cruda (CNN) autogenerados desde Python\nconst float signal_mean = {signal_mean:.9f}f;\nconst float signal_std = {signal_std:.9f}f;\n\n#endif // CNN_SCALER_PARAMS_H\n"""
with open(cnn_scaler_path, "w", encoding="utf-8") as f:
    f.write(cnn_scaler_content)
print(f"✅ Archivo de escala CNN autogenerado: {cnn_scaler_path}")

✅ Archivo de cabecera creado: /home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/include/NN_model.h


✅ Archivo de definición creado: /home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/src/NN_model.cpp


✅ Archivo de escala CNN autogenerado: /home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/include/cnn_scaler_params.h
